# CS431 — Đánh giá trên 5 cảnh giữ lại (Held-out Evaluation)

- **Cảnh giữ lại (5 cảnh chưa dùng ở bước pilot)**: `GearWorks`, `Laboratory`, `MilkCookie`, `Popcorn`, `Sculpture`.
- **Tham số chốt cố định từ bước pilot v2** (không tinh chỉnh thêm):
  - Mô hình: `depth-anything/DA3-BASE` (`f4a6c9b3c95e41c82048423d3493a81ec3fa810e`), `resolution = 504`, `4` ảnh/cảnh.
  - Hệ tọa độ: `colmap_sparse + colmap2world.txt` + căn chỉnh Sim(3) kết hợp góc xoay (`align_poses_sim3`).
  - Bộ chọn ảnh tham chiếu RGB-only (`cs431/reference.py` v2): `exposure`, `exposure_bridge`, `overlap` (gamma-CLAHE SIFT), `combined` (`0.5 * norm(exposure_bridge) + 0.5 * norm(overlap)`), so với `saddle_balanced` (DA3 mặc định) và lượt quét đầy đủ `sweep` (`oracle_f1`).


In [ ]:
import csv
import json
import os
import re
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path
from time import perf_counter

import numpy as np
from PIL import Image

WORK_DIR = Path('/kaggle/working')
OUT_DIR = WORK_DIR / 'outputs'
TMP_DIR = Path('/tmp/cs431_heldout')
REPO_DIR = TMP_DIR / 'Depth-Anything-3'
DATA_DIR = TMP_DIR / 'realx3d'
PC_DIR = TMP_DIR / 'pointclouds'

DA3_REPO_URL = 'https://github.com/ByteDance-Seed/Depth-Anything-3.git'
DA3_COMMIT = '3d835ec1a5802d64a8b8b15f817a1ab54809bfe4'
HF_REALX3D_BASE = 'https://huggingface.co/datasets/ToferFish/RealX3D/resolve/main'
VARYEXP_TAR_URL = f'{HF_REALX3D_BASE}/data_4/data4_varyexp.tar.gz'

HELDOUT_SCENES = ['GearWorks', 'Laboratory', 'MilkCookie', 'Popcorn', 'Sculpture']
NUM_VIEWS = 4
RESOLUTION = 504
COORDINATE_MODE = 'colmap_sparse + c2w'

for d in (OUT_DIR, TMP_DIR, DATA_DIR, PC_DIR):
    d.mkdir(parents=True, exist_ok=True)

def run_cmd(cmd, cwd=None, env=None, check=True):
    print(f"$ {' '.join(map(str, cmd))}", flush=True)
    return subprocess.run(cmd, cwd=cwd, env=env, check=check)

def locate_cs431_code() -> Path:
    for found in Path('/kaggle/input').rglob('run_experiment.py'):
        return found.parent
    raise FileNotFoundError('Could not locate cs431/run_experiment.py under /kaggle/input')

run_cmd(['nvidia-smi'], check=False)
if not REPO_DIR.exists():
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    run_cmd(['git', 'init'], cwd=REPO_DIR)
    run_cmd(['git', 'remote', 'add', 'origin', DA3_REPO_URL], cwd=REPO_DIR)
    run_cmd(['git', 'fetch', '--depth', '1', 'origin', DA3_COMMIT], cwd=REPO_DIR)
    run_cmd(['git', 'checkout', 'FETCH_HEAD'], cwd=REPO_DIR)

cs431_src = locate_cs431_code()
cs431_dst = REPO_DIR / 'cs431'
if cs431_dst.exists():
    shutil.rmtree(cs431_dst)
shutil.copytree(cs431_src, cs431_dst)

pkgs = [
    'addict', 'omegaconf', 'einops>=0.8', 'huggingface_hub', 'safetensors',
    'evo', 'plyfile', 'trimesh', 'e3nn', 'moviepy==1.0.3', 'imageio',
    'pycolmap', 'pillow_heif'
]
run_cmd([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

os.environ['HF_HOME'] = str(TMP_DIR / 'hf_cache')
for p in (str(REPO_DIR / 'src'), str(REPO_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

env = os.environ.copy()
env['PYTHONPATH'] = f"{REPO_DIR / 'src'}:{REPO_DIR}"
run_cmd([sys.executable, '-m', 'unittest', 'cs431.test_experiment', '-v'], cwd=REPO_DIR, env=env)


## Bước 2: Stream dữ liệu 5 cảnh giữ lại từ RealX3D


In [ ]:
def stream_extract_varyexp(target_scenes: list[str]):
    target_set = set(target_scenes)
    extracted_files = 0
    extracted_bytes = 0
    t0 = perf_counter()
    print(f'Streaming {VARYEXP_TAR_URL} for held-out scenes: {target_scenes}', flush=True)
    proc = subprocess.Popen(
        ['curl', '-fsSL', '--retry', '5', '--retry-delay', '5', VARYEXP_TAR_URL],
        stdout=subprocess.PIPE,
    )
    try:
        with tarfile.open(fileobj=proc.stdout, mode='r|gz') as tar:
            for member in tar:
                if not member.isfile():
                    continue
                parts = Path(member.name).parts
                if len(parts) < 3:
                    continue
                scene = parts[1] if parts[0] == 'varyexp' else parts[0]
                rel_parts = parts[2:] if parts[0] == 'varyexp' else parts[1:]
                if scene not in target_set or not rel_parts:
                    continue
                top = rel_parts[0]
                is_meta = top in ('transforms_train.json', 'colmap2world.txt')
                is_sparse = top == 'sparse'
                is_train_img = top.startswith('train') and member.name.lower().endswith(('.jpg', '.jpeg', '.png'))
                if not (is_meta or is_sparse or is_train_img):
                    continue
                dst = DATA_DIR / scene / Path(*rel_parts)
                dst.parent.mkdir(parents=True, exist_ok=True)
                src_f = tar.extractfile(member)
                if src_f is not None:
                    with dst.open('wb') as out_f:
                        shutil.copyfileobj(src_f, out_f)
                    extracted_files += 1
                    extracted_bytes += member.size
                    if extracted_files % 100 == 0:
                        print(f'  extracted {extracted_files} files ({extracted_bytes / 1e9:.2f} GB) in {perf_counter() - t0:.1f}s', flush=True)
    finally:
        if proc.stdout:
            proc.stdout.close()
        ret = proc.wait()
        if ret != 0:
            raise RuntimeError(f'curl exited with status {ret}')
    print(f'Done streaming held-out scenes: {extracted_files} files ({extracted_bytes / 1e9:.2f} GB) in {perf_counter() - t0:.1f}s', flush=True)

def download_pointcloud(scene: str) -> Path:
    url = f'{HF_REALX3D_BASE}/pointclouds/{scene}.tar.gz'
    scene_pc_dir = PC_DIR / scene
    scene_pc_dir.mkdir(parents=True, exist_ok=True)
    ply_path = None
    print(f'Streaming pointcloud {url}', flush=True)
    proc = subprocess.Popen(
        ['curl', '-fsSL', '--retry', '5', '--retry-delay', '5', url],
        stdout=subprocess.PIPE,
    )
    try:
        with tarfile.open(fileobj=proc.stdout, mode='r|gz') as tar:
            for member in tar:
                if not member.isfile():
                    continue
                if Path(member.name).name == 'cull_pointcloud.ply':
                    dst = scene_pc_dir / 'cull_pointcloud.ply'
                    src_f = tar.extractfile(member)
                    if src_f is not None:
                        with dst.open('wb') as out_f:
                            shutil.copyfileobj(src_f, out_f)
                    ply_path = dst
    finally:
        if proc.stdout:
            proc.stdout.close()
        ret = proc.wait()
        if ret != 0:
            raise RuntimeError(f'curl failed for {url} (exit {ret})')
    if ply_path is None:
        raise FileNotFoundError(f'cull_pointcloud.ply not found in {url}')
    return ply_path

stream_extract_varyexp(HELDOUT_SCENES)


## Bước 3: Chạy đánh giá trên 5 cảnh giữ lại × 3 điều kiện (`--sweep`)


In [ ]:
import torch
from plyfile import PlyData
from depth_anything_3.api import DepthAnything3
from cs431.geometry import cloud_metrics, predicted_poses, relative_pose_metrics, to_metric_poses, voxel_sample
from cs431.run_experiment import DA3_BASE, DA3_BASE_REVISION, evaluate_views, load_colmap_views

device = torch.device('cuda:0')
assert torch.cuda.is_available(), 'CUDA is required'
print('Loading DA3-Base on', torch.cuda.get_device_name(0), flush=True)
model = DepthAnything3.from_pretrained(DA3_BASE, revision=DA3_BASE_REVISION).to(device).eval()

def load_gt_cloud(ply_path: Path) -> np.ndarray:
    vertices = PlyData.read(str(ply_path))['vertex'].data
    xyz = np.column_stack([vertices[a] for a in 'xyz']).astype(float)
    return voxel_sample(xyz)

def _exposure_sort_key(folder_name: str):
    m = re.search(r'(\d+)', folder_name)
    return (0, int(m.group(1))) if m else (1, folder_name)

def build_scene_conditions(scene_dir: Path, num_views: int = NUM_VIEWS):
    transforms_path = scene_dir / 'transforms_train.json'
    frames = json.loads(transforms_path.read_text(encoding='utf-8'))['frames']
    idxs = np.linspace(0, len(frames) - 1, num_views, dtype=int)
    sampled_frames = [frames[i] for i in idxs]
    view_names = [Path(f['file_path']).name for f in sampled_frames]
    train_dirs = sorted(
        [p for p in scene_dir.iterdir() if p.is_dir() and p.name.startswith('train')],
        key=lambda p: _exposure_sort_key(p.name),
    )
    iso_dirs = [p for p in train_dirs if re.search(r'\d+', p.name)]
    conditions = {}
    default_paths = [(scene_dir / f['file_path']) for f in sampled_frames]
    if all(p.is_file() for p in default_paths):
        conditions['default_json'] = default_paths
    complete_iso = [d for d in iso_dirs if all((d / name).is_file() for name in view_names)]
    if len(complete_iso) >= 2:
        pick_indices = np.linspace(0, len(complete_iso) - 1, num_views, dtype=int)
        conditions['varyexp_mixed'] = [complete_iso[pi] / name for pi, name in zip(pick_indices, view_names)]
        mid_dir = complete_iso[len(complete_iso) // 2]
        conditions['uniform_mid'] = [mid_dir / name for name in view_names]
    return conditions

summary_rows = []
for scene in HELDOUT_SCENES:
    scene_dir = DATA_DIR / scene
    ply_path = download_pointcloud(scene)
    gt_points = load_gt_cloud(ply_path)
    c2w_mat = np.loadtxt(scene_dir / 'colmap2world.txt')
    conditions = build_scene_conditions(scene_dir, NUM_VIEWS)

    for cond_name, img_paths in conditions.items():
        paths, gt_poses = load_colmap_views(scene_dir / 'sparse' / '0', img_paths)
        gt_metric_poses = to_metric_poses(gt_poses, c2w_mat)
        images = [Image.open(p).convert('RGB') for p in paths]

        def infer(ordered, strategy, order):
            torch.cuda.synchronize(device)
            torch.cuda.reset_peak_memory_stats(device)
            t0 = perf_counter()
            prediction = model.inference(ordered, ref_view_strategy=strategy, process_res=RESOLUTION)
            torch.cuda.synchronize(device)
            res = {
                'inference_seconds': perf_counter() - t0,
                'peak_cuda_mib': torch.cuda.max_memory_allocated(device) / 1024**2,
            }
            t1 = perf_counter()
            res.update(relative_pose_metrics(predicted_poses(prediction.extrinsics), gt_poses[order]))
            res.update(cloud_metrics(prediction, gt_metric_poses[order], gt_points))
            res['metric_seconds'] = perf_counter() - t1
            return res

        report = {
            'model': DA3_BASE,
            'model_revision': DA3_BASE_REVISION,
            'coordinate_mode': COORDINATE_MODE,
            'device': str(device),
            'resolution': RESOLUTION,
            'images': [str(p) for p in paths],
            **evaluate_views(images, infer, sweep=True),
        }
        out_json = OUT_DIR / f'{scene}_{cond_name}_heldout.json'
        out_json.write_text(json.dumps(report, indent=2, allow_nan=False) + '\n', encoding='utf-8')

        sweep_f1s = [r['f1'] for r in report['sweep'] if r.get('f1') is not None]
        sweep_aucs = [r['auc10'] for r in report['sweep']]
        row = {
            'scene': scene,
            'condition': cond_name,
            'selection_s': round(report['selection_seconds'], 4),
        }
        for m in ('saddle_balanced', 'exposure', 'exposure_bridge', 'overlap', 'combined'):
            r = report['runs'][m]
            row[f'{m}_ref'] = r.get('reference_index')
            row[f'{m}_f1'] = round(r['f1'], 4) if r.get('f1') is not None else None
            row[f'{m}_auc10'] = round(r['auc10'], 4)
            row[f'{m}_med_m'] = round(r['median_pred_to_gt_m'], 4) if r.get('median_pred_to_gt_m') is not None else None
            row[f'{m}_s'] = round(r['inference_seconds'], 3)
        oracle = report.get('oracle_f1')
        row['oracle_f1'] = round(oracle['f1'], 4) if oracle else None
        row['span_f1'] = round(max(sweep_f1s) - min(sweep_f1s), 4) if sweep_f1s else None
        row['oracle_auc10'] = round(max(sweep_aucs), 4)
        row['span_auc10'] = round(max(sweep_aucs) - min(sweep_aucs), 4)
        summary_rows.append(row)
        print(
            f"{scene:12s} {cond_name:14s} | F1: def={row['saddle_balanced_f1']} comb={row['combined_f1']} brg={row['exposure_bridge_f1']} ovl={row['overlap_f1']} orc={row['oracle_f1']} | "
            f"AUC10: def={row['saddle_balanced_auc10']} comb={row['combined_auc10']} orc={row['oracle_auc10']}",
            flush=True,
        )

    shutil.rmtree(PC_DIR / scene, ignore_errors=True)


## Bước 4: Tổng hợp kết quả 5 cảnh giữ lại


In [ ]:
csv_path = OUT_DIR / 'summary_heldout.csv'
with csv_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader()
    writer.writerows(summary_rows)

md = [
    f'# CS431 Held-out Evaluation Results (5 scenes, {COORDINATE_MODE})',
    '',
    '## Per-scene results',
    '| Scene | Condition | Default F1 | Exposure F1 | Bridge F1 | Overlap F1 | Combined F1 | Oracle F1 | Span F1 | Default AUC10 | Combined AUC10 |',
    '| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |',
]
for r in summary_rows:
    md.append(
        f"| {r['scene']} | {r['condition']} | {r['saddle_balanced_f1']} | {r['exposure_f1']} | "
        f"{r['exposure_bridge_f1']} | {r['overlap_f1']} | {r['combined_f1']} | {r['oracle_f1']} | "
        f"{r['span_f1']} | {r['saddle_balanced_auc10']} | {r['combined_auc10']} |"
    )

md += ['', '## Mean by condition (5 held-out scenes)', '| Condition | Default F1 | Exposure F1 | Bridge F1 | Overlap F1 | Combined F1 | Δ(Comb-Def) F1 | Oracle F1 | Span F1 | Default AUC10 | Combined AUC10 | Δ(Comb-Def) AUC10 |', '| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |']
for cond in ('default_json', 'varyexp_mixed', 'uniform_mid'):
    sub = [r for r in summary_rows if r['condition'] == cond]
    if not sub:
        continue
    def mean_k(k):
        vals = [r[k] for r in sub if r[k] is not None]
        return round(float(np.mean(vals)), 4) if vals else None
    df1, cf1 = mean_k('saddle_balanced_f1'), mean_k('combined_f1')
    da, ca = mean_k('saddle_balanced_auc10'), mean_k('combined_auc10')
    md.append(
        f"| {cond} | {df1} | {mean_k('exposure_f1')} | {mean_k('exposure_bridge_f1')} | "
        f"{mean_k('overlap_f1')} | {cf1} | {round(cf1 - df1, 4) if df1 is not None and cf1 is not None else None} | "
        f"{mean_k('oracle_f1')} | {mean_k('span_f1')} | {da} | {ca} | {round(ca - da, 4)} |"
    )

md_text = '\n'.join(md) + '\n'
(OUT_DIR / 'summary_heldout.md').write_text(md_text, encoding='utf-8')
print(md_text)
